# 문제 1-3 : FewShotPromptTemplate과 시스템 메시지 활용 - 뉴스 키워드 추출기

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
PROVIDER = "groq"
CFG = {
    "groq": {"key": os.getenv("GROQ_API_KEY"), "url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "openai": {"key": os.getenv("OPENAI_API_KEY"), "url": None, "model": "gpt-4o-mini"},
}
c = CFG[PROVIDER]
if not c["key"]:
    raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")

llm = ChatOpenAI(api_key=c["key"], base_url=c["url"], model=c["model"], temperature=0)

examples = [
    {
        "news": "삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.",
        "keywords": "삼성전자, 인공지능, 엔비디아",
    },
    {
        "news": "세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.",
        "keywords": "세계보건기구, 건강위기, 국제협력",
    },
    {
        "news": "손흥민이 어제 열린 프리미어리그 경기에서 멀티골을 터뜨리며 팀의 3대 1 승리를 이끌었다. 이번 활약으로 시즌 득점 순위 상위권에 올라섰다.",
        "keywords": "손흥민, 프리미어리그, 멀티골",
    },
]

ep = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}"),
])

fs = FewShotChatMessagePromptTemplate(example_prompt=ep, examples=examples)

fp = ChatPromptTemplate.from_messages([
    ("system", "당신은 뉴스 키워드 추출 전문가입니다. 뉴스에서 핵심 키워드 3개를 추출하세요. "
               "예시와 똑같은 형식(키워드: 키워드1, 키워드2, 키워드3)으로만 답하고, 다른 설명은 덧붙이지 마세요."),
    fs,
    ("human", "{input}"),
])

chain = fp | llm | StrOutputParser()

tests = [
    "제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI)에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.",
    "한국은행이 기준금리를 동결하기로 결정했다. 물가 상승률이 안정세를 보이고 있지만 가계부채 증가와 환율 불안이 지속되고 있어 당분간 신중한 통화정책을 이어갈 것이라고 밝혔다.",
    "서울시가 내년부터 대중교통 무제한 이용권 가격을 인하하고 이용 대상을 확대한다. 시민의 교통비 부담을 줄이고 탄소 배출을 감축하기 위한 정책이다.",
]
for i, t in enumerate(tests, 1):
    try:
        print(f"[테스트 {i}] {t[:40]}...")
        print(chain.invoke({"input": t}))
        print()
    except Exception as e:
        print("오류:", e)


[테스트 1] 제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI St...
키워드: 제미나이 2.0, 구글 AI 스튜디오, 멀티모달 입력

[테스트 2] 한국은행이 기준금리를 동결하기로 결정했다. 물가 상승률이 안정세를 보이고...
키워드: 한국은행, 기준금리, 가계부채

[테스트 3] 서울시가 내년부터 대중교통 무제한 이용권 가격을 인하하고 이용 대상을 확...
키워드: 서울시, 대중교통, 탄소감축

